# Feature Engineering

Builds modeling features on top of the raw `Match` table in `data/database.sqlite`.

Feature engineering cited from https://pubmed.ncbi.nlm.nih.gov/40989473/

Run `python scripts/download_data.py` from the repo root first so that `data/database.sqlite` exists.

First feature: the match result label from the home team's perspective — `Win` / `Draw` / `Defeat`. This is the target variable for the outcome predictor.

Second feature: overall player rating. For each player, use the closest rating from the FIFA metrics as the representative value.

Third feature: team strength. Evaluate both teams on overall strength based on an aggregation of the player ratings.

In [44]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = REPO_ROOT / "data" / "database.sqlite"
assert DB_PATH.exists(), (
    f"Missing {DB_PATH}. Run `python scripts/download_data.py` from the repo root."
)

conn = sqlite3.connect(DB_PATH)
print(f"Opened {DB_PATH} ({DB_PATH.stat().st_size / 1024**2:.1f} MB)")

Opened /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/database.sqlite (298.6 MB)


## Load matches

Load the columns needed for the result label plus `match_api_id` and the 22 player ID
columns required for the player-rating lookup in the next section.
The full table is 115 columns wide; we keep only what we need.

In [45]:
matches = pd.read_sql(
    """
    SELECT match_api_id, date,
           home_team_api_id, away_team_api_id,
           home_team_goal, away_team_goal,
           home_player_1,  home_player_2,  home_player_3,
           home_player_4,  home_player_5,  home_player_6,
           home_player_7,  home_player_8,  home_player_9,
           home_player_10, home_player_11,
           away_player_1,  away_player_2,  away_player_3,
           away_player_4,  away_player_5,  away_player_6,
           away_player_7,  away_player_8,  away_player_9,
           away_player_10, away_player_11
    FROM Match
    ORDER BY date;
    """,
    conn,
)
print(f"Loaded {len(matches):,} matches")
matches.head()

Loaded 25,979 matches


,match_api_id,date,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,home_player_1,home_player_2,home_player_3,home_player_4,home_player_5,home_player_6,home_player_7,home_player_8,home_player_9,home_player_10,home_player_11,away_player_1,away_player_2,away_player_3,away_player_4,away_player_5,away_player_6,away_player_7,away_player_8,away_player_9,away_player_10,away_player_11
0,486263,2008-07-18 00:00:00,10192,9931,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,486264,2008-07-19 00:00:00,9930,10179,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,486265,2008-07-20 00:00:00,10199,9824,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,486266,2008-07-20 00:00:00,7955,10243,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,486267,2008-07-23 00:00:00,9931,9956,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Add the `result` column

Labels are from the home team's perspective:

- Win: home goals > away goals
- Draw: home goals == away goals
- Defeat: home goals < away goals

This matches how the rest of the dataset is structured (e.g. the bookmaker odds columns `B365H/D/A` are also home-perspective), so the label lines up cleanly with the other features.

In [46]:
conditions = [
    matches["home_team_goal"] > matches["away_team_goal"],
    matches["home_team_goal"] == matches["away_team_goal"],
    matches["home_team_goal"] < matches["away_team_goal"],
]
matches["result"] = np.select(conditions, ["Win", "Draw", "Defeat"], default="")

# Integer encoding for sklearn-style models
matches["result_code"] = matches["result"].map({"Defeat": 0, "Draw": 1, "Win": 2})

matches[["date", "home_team_goal", "away_team_goal", "result", "result_code"]].head(10)

,date,home_team_goal,away_team_goal,result,result_code
0,2008-07-18 00:00:00,1,2,Defeat,0
1,2008-07-19 00:00:00,3,1,Win,2
2,2008-07-20 00:00:00,1,2,Defeat,0
3,2008-07-20 00:00:00,1,2,Defeat,0
4,2008-07-23 00:00:00,1,0,Win,2
5,2008-07-23 00:00:00,1,2,Defeat,0
6,2008-07-23 00:00:00,1,0,Win,2
7,2008-07-24 00:00:00,2,1,Win,2
8,2008-07-24 00:00:00,0,2,Defeat,0
9,2008-07-26 00:00:00,2,0,Win,2


## Sanity checks

Confirm no rows fell through the conditions and that the class distribution shows the expected home-field advantage (~46% Win / 25% Draw / 29% Defeat for European league football).

In [47]:
assert matches["result"].notna().all(), "Some matches were not labeled"

print("Class counts:")
print(matches["result"].value_counts())
print("\nClass proportions:")
print(matches["result"].value_counts(normalize=True).round(3))

Class counts:
result
Win       11917
Defeat     7466
Draw       6596
Name: count, dtype: int64

Class proportions:
result
Win       0.459
Defeat    0.287
Draw      0.254
Name: proportion, dtype: float64


## Obtain player ratings

Melt the 22 player ID columns into long format, then merge against Player_Attributes to obtain the most recent rating before the match date.

In [48]:
# define player id columns
player_cols = [f'home_player_{i}' for i in range(1, 12)] + \
              [f'away_player_{i}' for i in range(1, 12)]

# melt 22 player id columns into one row per player per match
match_players = (
    matches[['match_api_id', 'date'] + player_cols]
    .melt(id_vars=['match_api_id', 'date'], value_vars=player_cols, value_name='player_api_id')
    .dropna(subset=['player_api_id'])
    .astype({'player_api_id': int})
)

# load player attributes
player_attributes = pd.read_sql("SELECT player_api_id, date, overall_rating FROM Player_Attributes", conn)

# sort by date, required for merge_asof
match_players['date'] = pd.to_datetime(match_players['date'])
player_attributes['date'] = pd.to_datetime(player_attributes['date'])

match_players = match_players.sort_values('date')
player_attributes = player_attributes.sort_values('date')

# get the most recent rating before each match date
rated = pd.merge_asof(
    match_players,
    player_attributes,
    on='date',
    by='player_api_id',
    direction='backward'
)

## Build match_team_stats

Step 2: aggregate rated by match and side into team-level stats.

Step 3: merge back onto matches so each match row has home and away rating features.

In [49]:
# extract side (home or away) from the variable column
rated['side'] = rated['variable'].str.split('_player_').str[0]

match_team_stats = (
    rated.groupby(['match_api_id', 'side'])['overall_rating']
    .agg(
        avg_rating='mean',
        min_rating='min',
        max_rating='max',
        std_rating='std',
    )
    .unstack('side')
)

# flatten MultiIndex columns, e.g. (avg_rating, home) becomes home_avg_rating
match_team_stats.columns = [
    f'{side}_{stat}' for stat, side in match_team_stats.columns
]
match_team_stats = match_team_stats.reset_index()

print(match_team_stats.shape)
match_team_stats.head()

(25221, 9)


,match_api_id,away_avg_rating,home_avg_rating,away_min_rating,home_min_rating,away_max_rating,home_max_rating,away_std_rating,home_std_rating
0,483129,69.200000,70.000000,61.0,51.0,81.0,81.0,6.250333,8.921883
1,483130,69.454545,74.454545,58.0,65.0,77.0,83.0,5.354692,5.905313
2,483131,68.818182,64.909091,55.0,47.0,75.0,77.0,7.110811,9.300049
3,483132,67.181818,70.444444,58.0,61.0,75.0,76.0,4.707827,5.270463
4,483133,70.363636,78.818182,58.0,74.0,79.0,83.0,7.131237,3.250175


In [50]:
# merge team stats back onto the matches dataframe
matches = matches.merge(match_team_stats, on='match_api_id', how='left')

print(matches.shape)
matches[['match_api_id', 'result', 'home_avg_rating', 'away_avg_rating']].head(10)

(25979, 38)


,match_api_id,result,home_avg_rating,away_avg_rating
0,486263,Defeat,NaN,NaN
1,486264,Win,NaN,NaN
2,486265,Defeat,NaN,NaN
3,486266,Defeat,NaN,NaN
4,486267,Win,NaN,NaN
5,486268,Defeat,NaN,NaN
6,486269,Win,NaN,NaN
7,486270,Win,NaN,NaN
8,486271,Defeat,NaN,NaN
9,486272,Win,NaN,NaN


Some matches have no player lineup data recorded in the database, so they produce NaN ratings after the merge. These are dropped before modelling.

In [51]:
null_rate = matches['home_avg_rating'].isna().mean()
print(f"Matches with no rating data: {null_rate:.1%}")
print(f"Total matches: {len(matches)}, missing: {matches['home_avg_rating'].isna().sum()}")

matches = matches.dropna(subset=['home_avg_rating', 'away_avg_rating'])
print(f'{len(matches):,} matches retained with full rating data')

Matches with no rating data: 2.9%
Total matches: 25979, missing: 758
25,221 matches retained with full rating data


In [52]:
conn.close()